# 4a_QRF_MODEL — QRF Training & Artefact Export

Trains (or loads) the **Quantile Random Forest** model using optimised
parameters from `output/sweeps/qrf_params.json`.

| § | Stage | Notes |
|---|---|---|
| 1 | Config & imports | Shared constants, paths |
| 2 | Load reference parquet | IHFC_obs.parquet |
| 3 | Load params & obs_sel | From `qrf_params.json` |
| 4 | Train/test split + scaler | `RANDOM_SEED=42`, `TEST_FRAC=0.20` |
| 5 | Fit QRF | Resume from pickle if exists |
| 6 | Predict + conformal calibration | qhat on held-out cal slice |
| 7 | Centre-bias correction spline | PCHIP, cal-set only |
| 8 | Metrics | R², RMSE, MAE, PICP, Shannon H |
| 9 | Diagnostic figures | Scatter, residuals, PI width |
| 10 | Bundle artefacts | `qrf_artefacts.pkl` → `output/models/` |

**Outputs**
- `output/models/qrf_model.pkl`
- `output/models/qrf_artefacts.pkl`  ← loaded by `5_TARGETS`
- `output/models/qrf_metrics.csv`
- `fig/models/qrf_*.png`

> **Design note:** `obs_sel` is read from `qrf_params['features']` so
> it automatically reflects whichever feature set the sweep selected.
> Falls back to `obs_model` from `config.py` if the key is absent.
> The correction spline is fitted on the **calibration slice** (last 10 %
> of train), consistent across all three model notebooks.

## 1. Imports & constants

In [16]:
import sys, json, pickle, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import PchipInterpolator
from scipy.stats import entropy as scipy_entropy
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────────────────────
local_data = Path('data')
param_dir  = Path('output/sweeps')
model_dir  = Path('output/models')
fig_dir    = Path('fig/models')
model_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ── shared constants ───────────────────────────────────────────────────────────
TEST_FRAC       = 0.20
RANDOM_SEED     = 42
TARGET_COL      = 'q'
Q_CLIP_MIN      = 0.0
QUANTILES       = [0.05, 0.25, 0.50, 0.75, 0.95]
QUANTILES_DENSE = np.linspace(0.02, 0.98, 49).tolist()
HIST_BIN_W      = 0.010
CONFORMAL_ALPHA = 0.10
SPLINE_PCTLS    = 200   # number of percentile knots for correction spline
FIG_DPI         = 150

from config import *
print(f'obs_model features : {len(obs_model)}')
print(f'q_clip_max         : {q_clip_max} W/m²')
print(f'Test fraction      : {TEST_FRAC}')


obs_model features : 21
q_clip_max         : 0.35 W/m²
Test fraction      : 0.2


## 2. Load reference data

In [17]:
df = pd.read_parquet(local_data / 'IHFC_obs.parquet').copy()
print(f'Raw rows: {len(df)}')

df = df.dropna(subset=obs_model)
print(f'After dropna : {len(df)} rows')

if 'weight' in df.columns:
    w_all = df['weight'].values.astype(np.float32)
    w_all = w_all / w_all.mean()
    print(f'Weights  min={w_all.min():.4f}  max={w_all.max():.4f}  mean={w_all.mean():.4f}')
else:
    w_all = np.ones(len(df), dtype=np.float32)
    print('WARNING: no weight column — using uniform weights')


Raw rows: 30848
After dropna : 30847 rows
Weights  min=0.1123  max=3.5262  mean=1.0000


## 3. Load sweep parameters

In [18]:
from quantile_forest import RandomForestQuantileRegressor

with open(param_dir / 'qrf_params.json') as fp:
    PARAMS = json.load(fp)

# obs_sel: exactly the feature list recorded in the sweep JSON (single source of truth)
obs_sel = PARAMS['obs_sel']
assert isinstance(obs_sel, list) and obs_sel, 'obs_sel missing/empty in sweep JSON'
print(f'Loaded qrf_params.json')
print(f'  obs_sel ({len(obs_sel)} features): {obs_sel}')
print(f'  n_estimators     : {PARAMS["n_estimators"]}')
print(f'  max_depth        : {PARAMS["max_depth"]}')
print(f'  min_samples_leaf : {PARAMS["min_samples_leaf"]}')
print(f'  max_features     : {PARAMS["max_features"]}')


Loaded qrf_params.json
  obs_sel (22 features): ['BOUGUER', 'CRUST_RHO', 'CTD', 'GEOID', 'REVEAL_S70', 'FREE_AIR', 'REVEAL_S80', 'REVEAL_VP90VS60', 'REVEAL_VP60VS70', 'REVEAL_VP50VS80', 'DEM', 'LITH_RHO', 'EMAG2_LOG', 'MOHO', 'REVEAL_S90', 'REVEAL_P150', 'SEDIMENT', 'LAB', 'SI', 'REVEAL_S100', 'MOHO_GRAV', 'MAG_SEIS_MOHO']
  n_estimators     : 1000
  max_depth        : 50
  min_samples_leaf : 3
  max_features     : 0.7


## 4. Train/test split & StandardScaler

In [19]:
X_all = df[obs_sel].values.astype(np.float32)
y_all = df[TARGET_COL].values.astype(np.float32)

assert np.isfinite(X_all).all(), 'NaNs in X_all — check obs_sel'
assert np.isfinite(y_all).all(), 'NaNs in y_all'

X_tr_raw, X_te_raw, y_tr, y_te, w_tr, w_te = train_test_split(
    X_all, y_all, w_all, test_size=TEST_FRAC, random_state=RANDOM_SEED)

scaler = StandardScaler().fit(X_tr_raw)
X_tr   = scaler.transform(X_tr_raw).astype(np.float32)
X_te   = scaler.transform(X_te_raw).astype(np.float32)

# Held-out calibration slice (last 10 % of train — no additional leakage)
cal_frac = float(PARAMS.get('cal_frac', 0.10))
n_cal    = max(int(len(X_tr) * cal_frac), 100)
X_cal, y_cal, w_cal = X_tr[-n_cal:], y_tr[-n_cal:], w_tr[-n_cal:]

print(f'Train  : {len(y_tr):,}    Test : {len(y_te):,}    Cal slice : {n_cal:,}')
print(f'Features used : {len(obs_sel)}')


Train  : 24,677    Test : 6,170    Cal slice : 2,467
Features used : 22


## 5. Fit QRF

Loads from pickle if `qrf_model.pkl` already exists.
Safe to re-run — will not re-train unnecessarily.

In [ ]:
qrf_path = model_dir / 'qrf_model.pkl'
t0 = time.time()

# Load a cached model only if it matches the current feature set; otherwise retrain.
# This prevents a stale pickle (trained on a different obs_sel) from silently
# shadowing a freshly-swept params file.
_cached = None
if qrf_path.exists():
    with open(qrf_path, 'rb') as fp:
        _cached = pickle.load(fp)
    if int(getattr(_cached, 'n_features_in_', -1)) != len(obs_sel):
        print(f'QRF: cached model has {_cached.n_features_in_} features but obs_sel '
              f'has {len(obs_sel)} — discarding stale pickle and retraining.')
        _cached = None

if _cached is not None:
    print('QRF: loading saved model …')
    qrf = _cached
else:
    print('QRF: training …')
    qrf = RandomForestQuantileRegressor(
        n_estimators     = int(PARAMS['n_estimators']),
        max_depth        = PARAMS['max_depth'] if str(PARAMS['max_depth']) != 'None' else None,
        min_samples_leaf = int(PARAMS['min_samples_leaf']),
        max_features     = PARAMS['max_features'],
        n_jobs           = -1,
        random_state     = RANDOM_SEED,
    )
    nan_mask = ~np.isfinite(X_tr).all(axis=1)
    if nan_mask.any():
        print(f'  WARNING: dropping {nan_mask.sum()} NaN rows before fit')
    Xf, yf, wf = X_tr[~nan_mask], y_tr[~nan_mask], w_tr[~nan_mask]
    qrf.fit(Xf, yf, sample_weight=wf)
    with open(qrf_path, 'wb') as fp:
        pickle.dump(qrf, fp)
    print(f'  saved → {qrf_path}')

assert qrf.n_features_in_ == len(obs_sel), (
    f'model expects {qrf.n_features_in_} features but obs_sel has {len(obs_sel)}')
print(f'Wall time: {(time.time()-t0)/60:.1f} min')


## 6. Test-set predictions + conformal calibration

Conformal scores computed on the held-out calibration slice;
`qhat` adjusts the PI bounds to achieve the requested coverage.

In [21]:
print(f'obs_sel   : {len(obs_sel)} → {obs_sel}')
print(f'X_te      : {X_te.shape}')
print(f'qrf       : {qrf.n_features_in_} features')

obs_sel   : 22 → ['BOUGUER', 'CRUST_RHO', 'CTD', 'GEOID', 'REVEAL_S70', 'FREE_AIR', 'REVEAL_S80', 'REVEAL_VP90VS60', 'REVEAL_VP60VS70', 'REVEAL_VP50VS80', 'DEM', 'LITH_RHO', 'EMAG2_LOG', 'MOHO', 'REVEAL_S90', 'REVEAL_P150', 'SEDIMENT', 'LAB', 'SI', 'REVEAL_S100', 'MOHO_GRAV', 'MAG_SEIS_MOHO']
X_te      : (6170, 22)
qrf       : 21 features


In [ ]:
# ── state guard: rebuild the test/cal matrices if they drift from the model ──
# Protects against out-of-order / stale-kernel execution, where a leftover X_te
# from an earlier run no longer matches the current obs_sel and fitted model.
_nf_model = int(qrf.n_features_in_)
if X_te.shape[1] != _nf_model or len(obs_sel) != _nf_model:
    print(f'  rebuilding X_te/X_cal: X_te had {X_te.shape[1]} cols, '
          f'obs_sel has {len(obs_sel)}, model expects {_nf_model}')
    assert len(obs_sel) == _nf_model, (
        f'obs_sel ({len(obs_sel)}) disagrees with the fitted model ({_nf_model}). '
        f'Re-run the params cell — qrf_params.json and the model are out of sync.')
    _X_all = df[obs_sel].values.astype(np.float32)
    _Xtr_raw, _Xte_raw, y_tr, y_te, w_tr, w_te = train_test_split(
        _X_all, y_all, w_all, test_size=TEST_FRAC, random_state=RANDOM_SEED)
    X_tr = scaler.transform(_Xtr_raw).astype(np.float32)
    X_te = scaler.transform(_Xte_raw).astype(np.float32)
    n_cal = max(int(len(X_tr) * float(PARAMS.get('cal_frac', 0.10))), 100)
    X_cal, y_cal, w_cal = X_tr[-n_cal:], y_tr[-n_cal:], w_tr[-n_cal:]
assert X_te.shape[1] == _nf_model, f'X_te has {X_te.shape[1]} features, model expects {_nf_model}'

print('Predicting on test set …')
qrf_preds = qrf.predict(X_te, quantiles=QUANTILES)           # (n, 5)
qrf_dense = qrf.predict(X_te, quantiles=QUANTILES_DENSE)     # (n, 49)
qrf_q50   = qrf_preds[:, QUANTILES.index(0.50)]
qrf_q05   = qrf_preds[:, QUANTILES.index(0.05)]
qrf_q95   = qrf_preds[:, QUANTILES.index(0.95)]
qrf_mean  = qrf_dense.mean(axis=1)
qrf_std   = qrf_dense.std(axis=1)

# conformal calibration
cal_out    = qrf.predict(X_cal, quantiles=[0.05, 0.95])
cal_q05_qrf, cal_q95_qrf = cal_out[:, 0], cal_out[:, 1]
scores_qrf = np.maximum(cal_q05_qrf - y_cal, y_cal - cal_q95_qrf)
qhat_qrf   = np.quantile(scores_qrf,
                         (1 - CONFORMAL_ALPHA) * (1 + 1 / len(scores_qrf)))
qrf_conf_lo = np.clip(qrf_q05 - qhat_qrf, Q_CLIP_MIN, None)
qrf_conf_hi = np.clip(qrf_q95 + qhat_qrf, None, q_clip_max)
print(f'Conformal qhat = {qhat_qrf*1e3:.2f} mW/m²')


## 7. Correction spline helpers

In [ ]:
from lib.utils import weighted_percentile, build_correction_spline

def apply_spline(vals, spline):
    out    = np.full_like(vals, np.nan, dtype=np.float32)
    ok     = np.isfinite(vals)
    out[ok] = np.clip(spline(vals[ok]).astype(np.float32), Q_CLIP_MIN, q_clip_max)
    return out

print('Spline helpers defined.')


## 7a. Fit & apply correction spline

PCHIP monotone interpolation from predicted percentiles → observed
percentiles (both weighted). Corrects regression-to-the-mean bias.
Fitted exclusively on the calibration slice to avoid test leakage.

In [ ]:
print('Fitting correction spline on calibration set …')
cal_q50_raw = qrf.predict(X_cal, quantiles=[0.50])
if cal_q50_raw.ndim == 2: cal_q50_raw = cal_q50_raw[:, 0]

qrf_spline, qrf_pred_p, qrf_true_p = build_correction_spline(
    cal_q50_raw, y_cal, w_cal)



# apply to test set
qrf_q50_corr   = apply_spline(qrf_q50, qrf_spline)
qrf_q05_corr   = apply_spline(qrf_q05, qrf_spline)
qrf_q95_corr   = apply_spline(qrf_q95, qrf_spline)
qrf_mean_corr  = apply_spline(qrf_mean, qrf_spline)

# spline plot
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(qrf_pred_p*1e3, qrf_true_p*1e3, lw=1.5, color='#2196F3', label='Spline')
ax.plot([0, q_clip_max*1e3], [0, q_clip_max*1e3], 'k--', lw=0.8, label='1:1')
ax.set_xlabel('Predicted Q [mW/m²]'); ax.set_ylabel('Corrected Q [mW/m²]')
ax.set_title('QRF centre-bias correction spline (cal set)')
ax.legend(); fig.tight_layout()
fig.savefig(fig_dir / 'qrf_correction_spline.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show(); print(f'Saved {fig_dir}/qrf_correction_spline.png')


## 8. Metrics

In [ ]:
from lib.utils import empirical_entropy as _empirical_entropy
def empirical_entropy(vals, bin_width=HIST_BIN_W):
    return _empirical_entropy(vals, Q_CLIP_MIN, q_clip_max, bin_width)

def mean_bias(y_true, y_pred):
    valid = np.isfinite(y_pred) & np.isfinite(y_true)
    return float(np.mean(y_pred[valid] - y_true[valid]))

def eval_metrics(y_true, y_pred, y_lo=None, y_hi=None, label=''):
    valid = np.isfinite(y_pred) & np.isfinite(y_true)
    r2    = r2_score(y_true[valid], y_pred[valid])
    rmse  = float(np.sqrt(mean_squared_error(y_true[valid], y_pred[valid])))
    mae   = float(mean_absolute_error(y_true[valid], y_pred[valid]))
    bias  = mean_bias(y_true, y_pred)
    picp  = np.nan
    piw   = np.nan
    if y_lo is not None and y_hi is not None:
        cov  = np.isfinite(y_lo) & np.isfinite(y_hi)
        picp = float(np.mean((y_true[cov] >= y_lo[cov]) & (y_true[cov] <= y_hi[cov])))
        piw  = float(np.mean(y_hi[cov] - y_lo[cov]))
    H = empirical_entropy(y_pred)
    print(f'{label:30s}  R²={r2:.4f}  RMSE={rmse*1e3:.2f} mW/m²  MAE={mae*1e3:.2f}  '
          f'Bias={bias*1e3:.2f}  PICP={picp:.3f}  PI_w={piw*1e3:.1f}  H={H:.3f}')
    return dict(label=label, r2=r2, rmse_mW=rmse*1e3, mae_mW=mae*1e3,
                bias_mW=bias*1e3, picp=picp, pi_width_mW=piw*1e3,
                shannon_H=H, nan_frac=float((~valid).mean()))

print('Metrics helpers defined.')


In [ ]:
H_obs  = empirical_entropy(y_te)
m_raw  = eval_metrics(y_te, qrf_q50,       y_lo=qrf_q05, y_hi=qrf_q95,
                       label='QRF Q50 raw')
m_corr = eval_metrics(y_te, qrf_q50_corr,  y_lo=qrf_q05_corr, y_hi=qrf_q95_corr,
                       label='QRF Q50 corrected')
m_conf = eval_metrics(y_te, qrf_q50_corr,  y_lo=qrf_conf_lo, y_hi=qrf_conf_hi,
                       label='QRF Q50 corr+conformal')
print(f'Observed entropy : {H_obs:.3f} nat')

metrics_df = pd.DataFrame([m_raw, m_corr, m_conf]).round(4)
metrics_df.to_csv(model_dir / 'qrf_metrics.csv', index=False)
print(f'Saved {model_dir}/qrf_metrics.csv')


## 9. Diagnostic figures

In [ ]:
from lib.utils import scatter_residuals_fig as _scatter_residuals_fig
def scatter_residuals_fig(rows, suptitle, savepath):
    return _scatter_residuals_fig(rows, suptitle, savepath, Q_CLIP_MIN, q_clip_max, FIG_DPI)

print('Figure helper defined.')


In [ ]:
scatter_residuals_fig(
    rows=[(y_te, qrf_q50, qrf_q50_corr, 'QRF Q50', '#2196F3')],
    suptitle='QRF — held-out test set (20%)',
    savepath=fig_dir / 'qrf_scatter_residuals.png',
)

# interval width vs observed
fig, ax = plt.subplots(figsize=(7, 4))
pi_width = (qrf_q95_corr - qrf_q05_corr) * 1e3
ax.scatter(y_te * 1e3, pi_width, s=1, alpha=0.2, c='#2196F3', rasterized=True)
ax.set_xlabel('Observed Q [mW/m²]'); ax.set_ylabel('PI90 width [mW/m²]')
ax.set_title(f'QRF corrected PI90 width  mean={pi_width.mean():.1f} mW/m²')
fig.tight_layout()
fig.savefig(fig_dir / 'qrf_pi_width.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show(); print(f'Saved {fig_dir}/qrf_pi_width.png')


## 10. Bundle artefacts for `5a_TARGETS`

All objects needed for target-grid prediction bundled into a single
`qrf_artefacts.pkl`. `5_TARGETS` loads this with `pickle.load()`.

In [ ]:
with open(param_dir / 'qrf_params.json') as f:
    _qp = json.load(f)

artefacts = dict(
    model           = qrf,           # QRF estimator (canonical key 'model')
    obs_sel         = obs_sel,
    scaler          = scaler,
    spline          = qrf_spline,    # canonical key 'spline'
    qhat_qrf        = float(qhat_qrf),
    QUANTILES       = QUANTILES,
    Q_CLIP_MIN      = float(Q_CLIP_MIN),
    q_clip_max      = float(q_clip_max),
    CONFORMAL_ALPHA = float(CONFORMAL_ALPHA),
    PARAMS          = _qp,
)

bundle_path = model_dir / 'qrf_artefacts.pkl'
with open(bundle_path, 'wb') as fp:
    pickle.dump(artefacts, fp)
print(f'Bundle saved → {bundle_path}')
print('Keys:', list(artefacts.keys()))